In [ ]:
# Exception Objects
    # - Exceptions are class instances. When an exception is raised, an instance is propagated through exception processing and matched against except clauses.
    # - Flexible exception categories: You can create subclasses to make exception handling more specific without changing existing try statements.
    # - State information and behavior: Exception instances can store information in attributes and provide methods for use inside exception handlers.
    # - Reuse by inheritance: Exception classes can inherit and customize common behavior, such as error messages.

# Exception Classes
    # Superclass matching: An exception matches an except clause if the clause names its class or any superclass of that class.
    # Subclass matching: An except clause also catches all subclasses of the specified exception class.
    # Exception hierarchies: Superclasses represent general exception categories, while subclasses represent more specific exceptions.
    # Catching categories: By naming a general superclass in an except clause, you can catch all exceptions belonging to that category.

# Coding Exceptions Classes
    # The example demonstrates how exception classes can form a hierarchy, allowing one except clause to catch several related exception types.



%run categoric.py
# 2. Raising instances

    # 6. Important design point
    # Well-designed programs usually should not need to identify the exact exception raised. Calling methods of the exception instance
    # should automatically execute behavior appropriate to that exception.

# Why Exception Hierarchies?
    # using a common superclass for related exceptions is better than listing individual exceptions.

    # 1. Listing exceptions individually
    # You can catch several exceptions using a tuple
try:
    func()
except (General, Specific1, Specific2): # Catch any of these
    ...
    # This works for small programs. However, in large exception hierarchies, listing every exception becomes difficult to maintain.

    # 2. The maintenance problem
    # Suppose a library defines two exceptions

    # mathlib.py
class Divzero(Exception): pass
class Oflow(Exception): pass

    # Users catch both:

    # client.py
import mathlib
try:
    mathlib.func()
except (mathlib.Divzero, mathlib.Oflow):
    ...handle and recover...

# Later, the library adds another exception:
class Uflow(Exception): pass    # Now users must update every relevant except clause to include Uflow. This creates a maintenance problem.

    # 3. Why not use an empty except?
    # Users might try:
try:
    mathlib.func()
except: # Catch everything here (or catch Exception)
    ...handle and recover...

    # But this catches too much, including:
    # - Running out of memory.
    # - Keyboard interrupts (Ctrl+C).
    # - System exits.
    # - Programming mistakes in their own code.
    # These exceptions should not be mistakenly treated as errors from the library.
    # So, It is usually better to catch specific exceptions rather than catch everything.

    # 4. Solution: Common superclass
    # Instead of defining independent exception classes, organize them under one superclass:
# mathlib.py
class NumErr(Exception): pass
class Divzero(NumErr): pass
class Oflow(NumErr): pass

        # The hierarchy is:
        # BaseException
        #     └── Exception
        #           └── NumErr
        #                 ├── Divzero
        #                 └── Oflow
        # Exception is the base class, but it does not hold a list of exceptions.
        # Instead, built-in and user-defined exception classes inherit from it, forming an inheritance hierarchy.
        # So, Exception is not a container or list. It is a superclass in the exception inheritance tree.

    # Now users only need:
# client.py
import mathlib
try:
    mathlib.func()
except mathlib.NumErr:          # Because Divzero and Oflow inherit from NumErr, this catches both.
    ...handle and recover...

    # 5. Adding new exceptions
    # Later, the library can add
class Uflow(NumErr): pass
    # Users do not need to change their existing except mathlib.NumErr statements.
    # The new exception is automatically caught because it belongs to the same hierarchy.

# Built-in Exception Classes
    # Python's built-in exceptions are predefined classes organized into an inheritance hierarchy. This allows Python to catch exceptions either individually or by their general categories.
    # 1. BaseException: Topmost root
        # - The top-level superclass of all exceptions.
        # - Provides default printing and state-retention behavior.
        # - When an instance is printed, it displays the constructor arguments.
        # - Stores constructor arguments in the args attribute as a tuple.
        # - User-defined exceptions should normally inherit from Exception, not directly from BaseException.
    # 2. Exception: Root of user-defined exceptions
        # - An immediate subclass of BaseException.
        # - The superclass of nearly all built-in exceptions.
        # - Nearly all user-defined exceptions should inherit from it.
        # - except Exception catches application-related exceptions but allows system exit events to pass.
    # 3. ArithmeticError: Root of numeric errors
        # A subclass of Exception that groups numeric errors:
        # - OverflowError
        # - ZeroDivisionError
        # - FloatingPointError
    # 4. LookupError: Root of indexing errors
        # A subclass of Exception that groups:
        # - IndexError — sequence indexing errors.
        # - KeyError — mapping lookup errors.
    # 5. OSError: Root of I/O and system errors
        # A subclass of Exception that provides error details through attributes such as:
        # - errno
        # - strerror
        # - filename
        # Specific subclasses include:
        # - FileNotFoundError
        # - PermissionError
        # - TimeoutError

# Built-in Exception Categories
    # Python's built-in exception hierarchy allows you to catch exceptions at different levels of specificity.
    # 1. General vs. specific exceptions
        # Because ArithmeticError is the superclass of OverflowError and ZeroDivisionError:
        # - except ArithmeticError: catches all numeric errors in this category.
        # - except ZeroDivisionError: catches only division-by-zero errors.
    # 2. Using Exception as a catchall
        # Because Exception is the superclass of application-level exceptions, you can catch almost all application errors:
try:
    ...
except Exception: # Exits not caught here
    ...handle all application exceptions...
else:
    ...handle no-exception case...

        # except Exception catches application-level exceptions.
        # System exit exceptions are not caught and can propagate normally.
        # else executes when no exception occurs in the try block.
    # 3. Disadvantages
        # Using Exception as a catchall has similar problems to an empty except:
        # - It may catch exceptions intended for other parts of the program.
        # - It may hide genuine programming errors.

# Default Printing and State
    # Built-in exceptions automatically provide two things:
    # 1. State retention: Constructor arguments are saved in the instance's args tuple.
    # 2. Default printing: Constructor arguments are automatically displayed when the exception instance is printed.

    # 1. No constructor arguments
raise IndexError # Same as IndexError(). No arguments means an empty args tuple and no display message.
    # 2. One constructor argument
raise IndexError('bad') # The argument 'bad' is saved in args. # IndexError: bad
    # 3. Multiple constructor arguments
i = IndexError('bad', 'stuff')
i.args      # ('bad', 'stuff')
print(i)    # ('bad', 'stuff')
i           # IndexError('bad', 'stuff')

    # Notice:
    # - args stores all arguments as a tuple.
    # - print(i) uses __str__.
    # - Displaying i in the interactive interpreter uses __repr__.

    # 4. User-defined exceptions
    # The same behavior applies to user-defined exceptions because they inherit the constructor and display methods from built-in exception classes.
class E(Exception): pass

i = E('bad', 'stuff')
i.args      # ('bad', 'stuff')
print(i)    # ('bad', 'stuff')
i           # E('bad', 'stuff')
    # 5. Accessing exception information in except
try:
    raise E('bad')  # Displays + saves constructor args
except E as X:
    print(f'{X} - {X.args} - {X!r}') # bad - ('bad',) - E('bad')
    # Here:
    # - X is the exception instance.
    # - {X} displays its string representation using __str__.
    # - {X.args} displays its saved constructor arguments.
    # - {X!r} displays its representation using __repr__.
# With multiple arguments:
try:
    raise E('bad', 'stuff')          # Multiple args save/display a tuple
except E as X:
    print(f'{X} - {X.args} - {X!r}') # ('bad', 'stuff') - ('bad', 'stuff') - E('bad', 'stuff')

# 6. Customizing behavior
    # Exception instances are not strings themselves. They use __str__ and __repr__ to produce display strings.
    # If you need different printing or state-retention behavior, you can redefine inherited methods such as __str__ and __init__.

# Custom Print Displays
    # By default, exception instances display the arguments passed to their constructors when printed or when an uncaught exception appears in an error message.
    # 1. Default display
class MyBad(Exception): pass

try:
    raise MyBad('Sorry--my mistake!')
except MyBad as X:
    print(X)     # Sorry--my mistake!
    # If the exception is not caught, the same message appears in the traceback:
    # MyBad: Sorry--my mistake!

    # 2. Custom display using __str__
    # You can customize the displayed message by redefining __str__ or __repr__.
class MyBad(Exception):
    def __str__(self):
        return 'Stuff happens...'
    # Now, when printed or shown in an uncaught exception, it displays:
    # Stuff happens...
    # The method can return a hardcoded string or process information stored in the exception instance.

    # 3. Why prefer __str__ over __repr__?
    # Built-in exception superclasses already have a __str__ method.
    # If you redefine only __repr__:
class Oops(Exception):
    def __repr__(self):
        return 'Custom display not used'

raise Oops("Nobody's perfect")  # Oops: Nobody's perfect
    # Your custom __repr__ is not used because Python prefers the inherited __str__ in this context.
    # But if you redefine __str__:
class Oops(Exception):
    def __str__(self):
        return 'Custom display used'

raise Oops("Nobody's perfect")  # Oops: Custom display used
    # Python's default exception display automatically combines two things:
    # 1. Exception class name: Oops
    # 2. Message returned by __str__: Custom display used

# Custom State and Behavior
    # Exception classes provide two additional capabilities:
    # 1. Custom state information: You can store extra information in instance attributes. The default constructor saves arguments in the args tuple,
    # but you can define your own constructor (__init__) for more specific needs.
    # 2. Custom behavior: You can define methods in exception classes to provide processing logic that can be used inside exception handlers.

# Providing Exception Details
    # how exception instances can carry additional information from the place where an error occurs to the place where it is handled.

    # 1. Why store information in exceptions?
    # - An exception can cross different module files.
    # - The raise statement and the try statement may be in separate files.
    # - Using global variables is unreliable because the handler may not know where those globals are located.
    # - Storing extra information inside the exception instance allows the handler to access it directly

    # 2. Using a custom constructor
    # Example:
class FormatError(Exception):
    def __init__(self, line, file): # Custom constructor
        self.line = line
        self.file = file

def parser(file): # Parse the file first
    raise FormatError(62, file=file) # When an error is found

try:
    parser('code.py')
except FormatError as X:
    print(f'Error at: {X.file} #{X.line}') # Custom state info  # Error at: code.py #62
    # How it works:
    # - FormatError(62, file=file) creates an exception instance.
    # - __init__ stores 62 in self.line and the filename in self.file.
    # - raise passes the instance along with the exception.
    # - except FormatError as X gives the handler access to that instance.
    # - X.file and X.line retrieve the stored information.

    # 3. Using the inherited constructor instead
    # You can also use the default constructor inherited from Exception:
class FormatError(Exception): pass # Inherited constructor

def parser(file):
    raise FormatError(file, 62) # No keywords allowed!

try:
    parser('code.py')
except FormatError as X:
    print(f'Error at: {X.args[0]} #{X.args[1]}') # Generic state info # Error at: code.py #62
    # Here:
    # - The inherited constructor stores the arguments in X.args.
    # - X.args[0] contains the filename.
    # - X.args[1] contains the line number.
    # - Unlike the custom constructor, this approach uses positional arguments rather than the keyword argument shown earlier.

# Providing Exception Methods
    # 1. Defining a method in an exception class
%run python3 parsely.py
%run python3 parsely.py
%run cat parser-errors.txt
    # 2. Inheritance and overriding methods
    # Exception classes can inherit methods and customize them.
class CustomFormatError(FormatError):
    def logerror(self):
        ...something unique here...
    # If this exception is raised:
raise CustomFormatError(...)
# And caught by:
except FormatError as exc:
    exc.logerror()          # Runs the raised class's version
    # The overridden logerror() method of CustomFormatError is executed.
    # This works because CustomFormatError is a subclass of FormatError

    # 3. Accessing the exception through sys.exc_info()
    # The raised exception instance assigned to exc is also available as the second item in the tuple returned by:
sys.exc_info()
    # This is useful when you do not specify an exception name in the except clause but still need access to the exception instance, its attributes, or methods.

    # 4. Using the traceback module
    # Instead of writing a custom message to a logfile, the logerror() method could generate Python's standard error message with a stack trace using the traceback standard-library module

# Exception Groups: Yet Another Star!
    # Python 3.11 introduced exception groups, which allow multiple exceptions to be raised and handled in one try statement.
    # 1. Normal except vs. except*
    # Normal except: handles one matching exception.
try:
    raise IndexError()
except IndexError:
    print('Got IE')
except (SyntaxError, TypeError):
    print('Got SE')

# Only one matching handler runs.

# except*: handles multiple exceptions from an exception group.
try:
    raise ExceptionGroup('Many', [IndexError(), SyntaxError()])
except* IndexError:
    print('Got IE')
except* (SyntaxError, TypeError):
    print('Got SE')     # Got IE Got SE
    # Both handlers run because the group contains two different exceptions.
# 2. Creating an exception group
ExceptionGroup('Many', [IndexError(), SyntaxError()])
    # It has two parts:
    # - 'Many' — a message describing the group.
    # - [IndexError(), SyntaxError()] — the exceptions inside the group.
# 3. How except* processes exceptions
    # - Each except* clause executes at most once.
    # - It handles all exceptions matching its type.
    # - Each exception is handled by only the first matching clause.
    # - Unmatched exceptions are raised again after the handlers finish.
    # - as E stores the matching exceptions as an exception group in E.
# For example:
excs = ExceptionGroup('Many', [
    IndexError(), SyntaxError(), TypeError()
])

try:
    raise excs
except* IndexError as E:
    print(E.exceptions)
except* SyntaxError as E:
    print(E.exceptions)
    # The first handler catches IndexError, and the second catches SyntaxError.
    # TypeError is unmatched, so it is raised again.
# 4. Multiple exceptions of the same type
    # One handler can catch several exceptions of the same type:
try:
    raise ExceptionGroup('Dups', [
        IndexError(), TypeError(1), TypeError(2)
    ])
except* IndexError:
    print('Got IE')
except* TypeError as E:
    print(E.exceptions)
    # The TypeError handler receives both TypeError(1) and TypeError(2).
# 5. Order of handlers matters
    # An exception also matches a superclass.
    # For example, IndexError is a subclass of LookupError.
try:
    raise ExceptionGroup('Dups', [
        IndexError(), TypeError(1), TypeError(2)
    ])
except* (TypeError, LookupError) as E:
    print('Got1:', E)
except* IndexError as E:
    print('Got2:', E)
    # The first handler catches all three exceptions:
    # - IndexError through its superclass LookupError.
    # - Both TypeError exceptions.
    # The second handler does not run because the IndexError has already been consumed
# 6. Can except* handle a normal exception?
    # Yes. Python automatically wraps an individual exception in a group.
try:
    raise IndexError
except* IndexError as E:
    print(E.exceptions)
    # The handler receives an exception group containing the IndexError.
# 7. Can normal except handle an exception group?
    # Yes. A normal except can catch the entire group and process its contents manually.
    # However, except* cannot catch an exception group as a whole
# 8. Important syntax restrictions
    # - Empty except* is not allowed.
    # - Normal except and except* cannot be mixed in the same try.
    # - else and finally can be used with except*.
    # - break, continue, and return are not allowed inside except*